# Calculate the probe correlations before and after correction

In [1]:
import pandas as pd
from scipy.stats import pearsonr

In [2]:
cancer = "LUSC"
metric = "CPE"

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)

df_cancer_meta = df_meta_test[df_meta_test["Cancer.type"] == cancer]
df_cancer_beta = df_beta_test.loc[df_cancer_meta["Barcode"]]

## Load adjusted beta values

In [4]:
monte = pd.read_parquet(f"../../data/monte_outputs/probe_correction/{cancer}_beta_correction.parquet")
monte = monte.loc[df_cancer_meta["Barcode"]]
monte_probe_ids = set(monte.columns)

In [5]:
purebeta = pd.read_parquet(
        f"../../data/cancer-methyl/purebeta_results/final_split/20251113_131504/{cancer}/{cancer}_beta_purified_20251113_131504.parquet"
    )
purebeta = purebeta.set_index("CpG").transpose()
purebeta = purebeta.loc[df_cancer_meta["Barcode"]]
purebeta_probe_ids = set(purebeta.columns)

In [6]:
infiniumPurify = pd.read_parquet(
        f"../../data/cancer-methyl/purified/infiniumpurify/{cancer}_beta_purified_20251113_113223.parquet"
    )
infiniumPurify = infiniumPurify.set_index("CpG").transpose()
infiniumPurify = infiniumPurify.loc[df_cancer_meta["Barcode"]]
infiniumPurify_probe_ids = set(infiniumPurify.columns)

In [7]:
intersect_probes = sorted(monte_probe_ids & purebeta_probe_ids & infiniumPurify_probe_ids)

In [8]:
df_cancer_beta = df_cancer_beta[intersect_probes]
monte = monte[intersect_probes]
purebeta = purebeta[intersect_probes]
infiniumPurify = infiniumPurify[intersect_probes]

## Calculate correlations between probe and purity

In [9]:
results = []
corr_group = []
for method, beta in zip(
    ["Unadjusted", "MONTE", "PureBeta", "InfiniumPurify"],
    [df_cancer_beta, monte, purebeta, infiniumPurify],
):
    for probe in intersect_probes:
        df_cancer_meta_subset = df_cancer_meta.dropna(subset=[metric])
        beta_subset = beta.loc[df_cancer_meta_subset["Barcode"], probe]
        metric_cor, _ = pearsonr(beta_subset, df_cancer_meta_subset[metric])
        results.append((method, probe, metric_cor))

In [10]:
df_results = pd.DataFrame(results, columns=["method", "probe", "metric_pearsonr"])
df_results["abs_metric_pearsonr"] = df_results["metric_pearsonr"].abs()

df_unadjusted = df_results[df_results["method"] == "Unadjusted"].copy()
df_unadjusted["corr_metric_group"] = pd.cut(
    df_unadjusted["abs_metric_pearsonr"],
    bins=[0.00, 0.25, 0.50, 0.75, 1.00],
    right=False,
    include_lowest=True,
    labels=["[0.00, 0.25)", "[0.25, 0.50)", "[0.50, 0.75)", "[0.75, 1.00)"],
)

df_results["corr_metric_group"] = df_results["probe"].map(
    df_unadjusted.set_index("probe")["corr_metric_group"]
)

In [11]:
df_results.to_csv(f"../../data/benchmark/benchmark_probe_corrections_{cancer}.csv", index=False)